# Build Step 2 — Operational Collections DDL (mongosh)

**Framework:** File Validation & Routing (`fvr` database) • **Deliverable:** the 8 Phase-1 operational collections with `$jsonSchema` validators + indexes, including every LLD v1.1/v1.2 field addition • **Prerequisite:** Build Step 1 (registry collections + seeds) already applied

## Run order & conventions

* Run each fenced block in `mongosh` against `use fvr`, in cell order. Re-runnable: `createCollection` guarded, `createIndex` idempotent.
* **Collections are created EMPTY** — the FastAPI app (build Step 3) is their only writer, using the guarded-atomic write patterns from the Resilience Addendum. The ONLY seeds here are `notifications` defaults.
* **This notebook supersedes the master design's `MongoDB Index & Schema Validation Specs` cell for these 8 collections** — it carries the v1.1 additions (`upload_sessions.status`, `file_registry.gridfs_purged_at/legal_hold`) and v1.2 (`validation_results.run_seq`) the master cell predates.
* GridFS: the default `fs` bucket is created lazily on first upload (driver `chunkSizeBytes: 261144`); only its secondary index is created here.

## Scope

| # | Collection | Key indexes | Notes |
| --- | --- | --- | --- |
| 1 | `event_log` | unique `(event_type, dedup_key)` (idempotent ingestion), partial DEAD scan, queue-depth scan | TTL proposed 30 d on PROCESSED only — **commented out pending compliance sign-off** |
| 2 | `upload_sessions` | user's sessions, `{status, started_at}` (janitor) | `status` enum = SESSION_STATUS (Resilience Addendum) |
| 3 | `file_registry` | unique `sha256_checksum`, status/source/session queries, **partial purge-candidate scan** | `gridfs_purged_at` + `legal_hold` (Purge Job Design) |
| 4 | `validation_results` | `{file_id, run_seq, executed_at}` | `run_seq` = re-validate discriminator (LLD v1.2) |
| 5 | `file_transfers` | **partial-unique `(file_id, direction)` WHERE COMPLETED** | never transferred twice per direction |
| 6 | `file_lifecycle` | `{file_id, timestamp}` | append-only location history |
| 7 | `audit_log` | `{entity_id, ts}` | app role must be **insert-only** (see security note in cell 4) |
| 8 | `notifications` | `{event, is_active}` | 3 seed alerts |
| — | `fs.files` | secondary `{"metadata.file_id": 1}` | GridFS reverse lookup |

**Field reference:** `Database Design LLD` v1.2 (authoritative). State-machine and enum dictionary: LLD §9.

## 1. DDL — `event_log`, `upload_sessions`

```javascript
use fvr

// ---------- event_log ---------- (canonical event store + queue)
if (!db.getCollectionInfos({name: "event_log"}).length) {
  db.createCollection("event_log", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["event_id", "event_type", "source_id", "detection_method", "dedup_key",
                 "payload", "processing_status", "retry_count", "received_at"],
      properties: {
        event_id:          { bsonType: "string", pattern: "^evt_" },
        event_type:        { enum: ["FILE_ARRIVED", "FILE_STAGED", "FILE_IDENTIFIED", "FILE_UNIDENTIFIED",
                                    "VALIDATION_COMPLETED", "FILE_ROUTED", "TRANSFER_COMPLETED",
                                    "TRANSFER_FAILED", "BRONZE_PROCESSED", "RECONCILIATION_COMPLETED",
                                    "FILE_ARCHIVED", "FILE_PURGED"] },
        source_id:         { bsonType: "string" },
        detection_method:  { enum: ["PUSH", "WEBHOOK", "WATCHER", "POLL"] },
        dedup_key:         { bsonType: "string", minLength: 1 },
        payload:           { bsonType: "object" },
        processing_status: { enum: ["RECEIVED", "PROCESSING", "PROCESSED", "DEAD"] },
        retry_count:       { bsonType: "int", minimum: 0 },
        last_error:        { bsonType: ["string", "null"] },
        correlation_id:    { bsonType: ["string", "null"] },
        received_at:       { bsonType: "date" }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.event_log.createIndex({ event_id: 1 }, { unique: true });
db.event_log.createIndex({ event_type: 1, dedup_key: 1 }, { unique: true });   // idempotent ingestion
db.event_log.createIndex({ processing_status: 1, received_at: 1 });            // worker scan + queue depth
db.event_log.createIndex(
  { received_at: 1 },
  { partialFilterExpression: { processing_status: "DEAD" } });                // DEAD-letter re-drive view
// TTL — 30 days on PROCESSED only. UNCOMMENT after compliance sign-off (HLD open item):
// db.event_log.createIndex(
//   { processed_at: 1 },
//   { expireAfterSeconds: 2592000, partialFilterExpression: { processing_status: "PROCESSED" } });

// ---------- upload_sessions ---------- (portal interaction record)
if (!db.getCollectionInfos({name: "upload_sessions"}).length) {
  db.createCollection("upload_sessions", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["session_id", "source_id", "user_id", "user_email", "client_ip", "roles_snapshot",
                 "files_received_count", "parts", "status", "started_at"],
      properties: {
        session_id:    { bsonType: "string", pattern: "^ses_" },
        source_id:    { bsonType: "string" },
        status:       { enum: ["IN_PROGRESS", "COMPLETED", "COMPLETED_PARTIAL",
                               "INCOMPLETE", "ABANDONED"] },                     // SESSION_STATUS (v1.1)
        user_id:      { bsonType: "string" },
        user_email:   { bsonType: "string" },
        client_ip:    { bsonType: "string" },
        machine_name: { bsonType: ["string", "null"] },
        user_agent:   { bsonType: ["string", "null"] },
        auth_method:  { bsonType: ["string", "null"] },
        roles_snapshot: { bsonType: "array", minItems: 1,
                          items: { enum: ["UPLOADER", "OPS", "ADMIN"] } },
        files_declared_count: { bsonType: ["int", "null"], minimum: 0 },
        files_received_count: { bsonType: "int", minimum: 0 },
        parts: { bsonType: "array", items: { bsonType: "object",
                 required: ["file_id", "status", "chunks_received"],
                 properties: {
                   file_id:         { bsonType: "string" },
                   gridfs_file_id:  { bsonType: ["objectId", "null"] },
                   status:          { enum: ["UPLOADING", "FINALIZED", "ABANDONED"] },
                   chunks_received: { bsonType: "int", minimum: 0 }
                 } } },
        started_at:   { bsonType: "date" },
        completed_at: { bsonType: ["date", "null"] }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.upload_sessions.createIndex({ session_id: 1 }, { unique: true });
db.upload_sessions.createIndex({ user_id: 1, started_at: -1 });     // my-uploads
db.upload_sessions.createIndex({ status: 1, started_at: 1 });       // session janitor scan (v1.1)
```

## 2. DDL — `file_registry`

```javascript
// ---------- file_registry ---------- (control entity + state machine — FK spine)
if (!db.getCollectionInfos({name: "file_registry"}).length) {
  db.createCollection("file_registry", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["file_id", "source_id", "original_name", "gridfs_file_id", "size_bytes",
                 "extension", "sha256_checksum", "current_status", "retry_count",
                 "received_at", "updated_at"],
      properties: {
        file_id:       { bsonType: "string", pattern: "^file_" },
        source_id:     { bsonType: "string" },
        original_name: { bsonType: "string", minLength: 1 },
        staged_path:   { bsonType: ["string", "null"] },
        gridfs_file_id: { bsonType: "objectId" },
        size_bytes:    { bsonType: ["long", "int"], minimum: 0 },
        extension:    { bsonType: "string" },
        mime_type:    { bsonType: ["string", "null"] },
        sha256_checksum: { bsonType: "string", minLength: 64, maxLength: 64 },
        file_type_id:       { bsonType: ["string", "null"] },
        schema_version_id:  { bsonType: ["string", "null"] },
        identification_method: { enum: ["EXPLICIT", "FILENAME_PATTERN", "HEADER_FINGERPRINT",
                                         "CONTENT_SNIFF", null] },
        identification_confidence: { bsonType: ["double", "int", "null"] },
        identification_evidence: { bsonType: ["object", "null"] },
        unidentified_reason: { bsonType: ["string", "null"] },
        upload_session_id: { bsonType: ["string", "null"] },
        uploaded_by:   { bsonType: ["string", "null"] },
        client_ip:     { bsonType: ["string", "null"] },
        machine_name:  { bsonType: ["string", "null"] },
        current_status: { enum: ["RECEIVED", "IDENTIFYING", "VALIDATING", "PASSED", "FAILED",
                                 "WARNED", "UNIDENTIFIED", "ROUTED_INTAKE", "ROUTED_QUARANTINE",
                                 "REJECTED", "FRAMEWORK_ERROR"] },
        batch_id:      { bsonType: ["string", "null"] },
        retry_count:   { bsonType: "int", minimum: 0 },
        received_at:   { bsonType: "date" },
        updated_at:    { bsonType: "date" },
        gridfs_purged_at: { bsonType: ["date", "null"] },                       // tombstone (v1.1)
        legal_hold:      { bsonType: ["bool", "null"] }                          // per-file override (v1.1)
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.file_registry.createIndex({ file_id: 1 }, { unique: true });
db.file_registry.createIndex({ sha256_checksum: 1 }, { unique: true });         // duplicate detection (C1)
db.file_registry.createIndex({ current_status: 1 });                             // ops board filter
db.file_registry.createIndex({ source_id: 1, received_at: -1 });                 // ops board time view
db.file_registry.createIndex({ upload_session_id: 1 });                          // my-uploads drill-down
// Purge candidate scan (Purge Job Design — Phase 3 job, index ready now)
db.file_registry.createIndex(
  { file_type_id: 1, current_status: 1, received_at: 1 },
  { partialFilterExpression: { gridfs_purged_at: null } });
```

## 3. DDL — `validation_results`, `file_transfers`, `file_lifecycle`

```javascript
// ---------- validation_results ---------- (rule outcomes with evidence — append-only incl. re-validates)
if (!db.getCollectionInfos({name: "validation_results"}).length) {
  db.createCollection("validation_results", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["result_id", "file_id", "run_seq", "rule_id", "rule_version",
                 "status", "details", "executed_at"],
      properties: {
        result_id:    { bsonType: "string", pattern: "^vr_" },
        file_id:     { bsonType: "string" },
        batch_id:    { bsonType: ["string", "null"] },
        run_seq:     { bsonType: "int", minimum: 1 },          // 1 = original run; 2,3… = re-validates (v1.2)
        rule_id:     { bsonType: "string" },
        rule_version: { bsonType: "int", minimum: 1 },
        status:      { enum: ["PASS", "FAIL", "WARN", "SKIPPED", "ERROR"] },
        details:     { bsonType: "object" },
        executed_at: { bsonType: "date" },
        duration_ms: { bsonType: ["int", "long", "null"], minimum: 0 }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.validation_results.createIndex({ result_id: 1 }, { unique: true });
db.validation_results.createIndex({ file_id: 1, run_seq: 1, executed_at: -1 });  // results screen + latest run

// ---------- file_transfers ---------- (routing manifest — one COMPLETED copy per direction)
if (!db.getCollectionInfos({name: "file_transfers"}).length) {
  db.createCollection("file_transfers", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["transfer_id", "file_id", "from_location", "to_location",
                 "direction", "status", "attempts", "source_sha256"],
      properties: {
        transfer_id:   { bsonType: "string", pattern: "^trf_" },
        file_id:      { bsonType: "string" },
        policy_id:   { bsonType: ["string", "null"] },
        from_location: { bsonType: "string" },
        to_location:   { bsonType: "string" },
        direction:   { enum: ["MONGO_TO_INTAKE", "MONGO_TO_QUARANTINE", "MONGO_TO_ARCHIVE"] },
        status:      { enum: ["PENDING", "IN_PROGRESS", "COMPLETED", "FAILED", "RETRYING"] },
        attempts:    { bsonType: "int", minimum: 0 },
        last_error:  { bsonType: ["string", "null"] },
        source_sha256: { bsonType: "string", minLength: 64, maxLength: 64 },
        dest_sha256:   { bsonType: ["string", "null"], minLength: 64, maxLength: 64 },
        bytes_written: { bsonType: ["long", "int", "null"], minimum: 0 },
        worker_id:   { bsonType: ["string", "null"] },
        started_at:  { bsonType: ["date", "null"] },
        completed_at: { bsonType: ["date", "null"] }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.file_transfers.createIndex({ transfer_id: 1 }, { unique: true });
// The structural idempotency guard (C4): a file can never land in a direction twice
db.file_transfers.createIndex(
  { file_id: 1, direction: 1 },
  { unique: true, partialFilterExpression: { status: "COMPLETED" } });
db.file_transfers.createIndex({ status: 1, started_at: 1 });   // retry scanner / backpressure metric

// ---------- file_lifecycle ---------- (physical location history — append-only)
if (!db.getCollectionInfos({name: "file_lifecycle"}).length) {
  db.createCollection("file_lifecycle", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["event_id", "file_id", "location", "timestamp"],
      properties: {
        event_id:          { bsonType: "string", pattern: "^flc_" },
        file_id:          { bsonType: "string" },
        location:         { enum: ["GRIDFS", "INTAKE_VOLUME", "QUARANTINE", "ARCHIVE", "PURGED"] },
        timestamp:        { bsonType: "date" },
        triggered_by_job: { bsonType: ["string", "null"] }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.file_lifecycle.createIndex({ event_id: 1 }, { unique: true });
db.file_lifecycle.createIndex({ file_id: 1, timestamp: 1 });
```

## 4. DDL — `audit_log`, `notifications` + GridFS index + alert seeds

```javascript
// ---------- audit_log ---------- (append-only trail — enforcement is role-based, see note)
if (!db.getCollectionInfos({name: "audit_log"}).length) {
  db.createCollection("audit_log", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["event_id", "ts", "actor", "entity_type", "entity_id", "action"],
      properties: {
        event_id:    { bsonType: "string", pattern: "^aud_" },
        ts:         { bsonType: "date" },
        actor:      { bsonType: "string", minLength: 1 },   // user_id or system:<component>
        entity_type: { bsonType: "string" },
        entity_id:  { bsonType: "string" },
        action:     { enum: ["CONFIG_CHANGE", "STATUS_CHANGE", "FILE_ROUTED", "RULE_ADDED",
                              "FILE_PURGED", "OTHER"] },
        before:     { bsonType: ["object", "null"] },
        after:      { bsonType: ["object", "null"] },
        approver:   { bsonType: ["string", "null"] },        // required on config changes (dual control)
        correlation_id: { bsonType: ["string", "null"] }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.audit_log.createIndex({ event_id: 1 }, { unique: true });
db.audit_log.createIndex({ entity_id: 1, ts: -1 });   // drill-down
db.audit_log.createIndex({ ts: 1 });                    // time-range audit pulls
// SECURITY NOTE: create the app role with insert-only privileges on this collection:
//   db.createRole("fvr_app_audit_writer",
//     { privileges: [{ resource: { db: "fvr", collection: "audit_log" },
//                      actions: ["insert", "find"] }], roles: [] })

// ---------- notifications ---------- (alert routing config)
if (!db.getCollectionInfos({name: "notifications"}).length) {
  db.createCollection("notifications", {
    validator: { $jsonSchema: {
      bsonType: "object",
      required: ["notification_id", "event", "channel", "recipients", "is_active"],
      properties: {
        notification_id: { bsonType: "string", pattern: "^not_" },
        event:     { bsonType: "string", minLength: 1 },
        condition: { bsonType: ["object", "null"] },
        channel:   { enum: ["email", "teams", "webhook"] },
        recipients: { bsonType: "array", minItems: 1, items: { bsonType: "string" } },
        is_active: { bsonType: "bool" }
      }
    }},
    validationLevel: "strict", validationAction: "error"
  });
}
db.notifications.createIndex({ notification_id: 1 }, { unique: true });
db.notifications.createIndex({ event: 1, is_active: 1 });

// ---------- GridFS secondary index ---------- (bucket fs is created on first upload)
db.fs.files.createIndex({ "metadata.file_id": 1 });

// ---------- notification seeds ---------- // EDIT recipients before running
db.notifications.insertMany([
  { notification_id: "not_transfer_failed", event: "TRANSFER_FAILED",
    condition: { min_attempts: 3 }, channel: "email",
    recipients: ["data-ops@yourcompany.com"], is_active: true },          // EDIT
  { notification_id: "not_framework_error", event: "FRAMEWORK_ERROR",
    condition: { min_retry_count: 3 }, channel: "email",
    recipients: ["data-ops@yourcompany.com"], is_active: true },          // EDIT
  { notification_id: "not_session_incomplete", event: "SESSION_INCOMPLETE",
    condition: {}, channel: "email",
    recipients: ["portal-support@yourcompany.com"], is_active: true }    // EDIT
], { ordered: false });
```

## 5. Verification — run after cells 1–4

```javascript
use fvr

// All 8 operational collections exist, are empty, and enforce at error level
["event_log","upload_sessions","file_registry","validation_results",
 "file_transfers","file_lifecycle","audit_log","notifications"].forEach(c => {
  const info = db.getCollectionInfos({ name: c })[0];
  print(c.padEnd(20), "docs:", String(db[c].countDocuments({})).padStart(4),
        "action:", info ? info.options.validationAction : "MISSING");
});
// expected: all 0 docs (notifications: 3) and validationAction: error for all

// Structural idempotency guards present on:
//   event_log        -> unique (event_type, dedup_key)
//   file_registry    -> unique sha256_checksum
//   file_transfers    -> partial-unique (file_id, direction) WHERE status COMPLETED
db.file_transfers.getIndexes().forEach(i =>
  printjson(i));   // confirm the partialFilterExpression on file_id+direction

// TTL state: should be ABSENT until compliance signs off (cell 1 has the commented line)
print("event_log TTL indexes:",
  db.event_log.getIndexes().filter(i => i.expireAfterSeconds).length);   // expected: 0

// Janitor + ops indexes present
db.upload_sessions.getIndexes().forEach(i => print(i.name));
db.file_registry.getIndexes().forEach(i => print(i.name));
```

**Pass criteria (Step 2):** all 8 collections report `validationAction: error`; zero unexpected documents; the three structural unique guards and the partial purge-candidate index exist; TTL count is 0 until compliance confirms the 30-day window.

## What comes next

* **Step 3 — FastAPI upload API** (manifest/parts/complete, GridFS stream, `FILE_ARRIVED` events) against these collections, then the pipeline worker (validation engine + rule plugins + routing) — checkpoint **R2** is the rule dry-run on real files.
* **GridFS writes** set `chunkSizeBytes: 261144` (255 KB) and the `fs.files` metadata contract from LLD §8.2 — enforced in application code, not DDL.
* **Write discipline from day one:** all worker writes use the four guarded-atomic patterns (claim / state-guard / `_v` optimistic / structural unique) from the Resilience Addendum — the schema above is what makes them safe.